In [1]:
import orjson
import numpy as np
from pathlib import Path
from datasets import Dataset, Features, Video, Sequence, Value, Array2D
from multiprocessing import cpu_count

In [2]:
def dataset_generator(folder_paths):
    for folder_path in folder_paths:
        folder_name = folder_path.name
        glob_pattern = f"{folder_name}_*.json"

        for json_path in sorted(folder_path.glob(glob_pattern)):
            video_path = json_path.with_suffix(".mp4")

            if not video_path.exists():
                raise FileNotFoundError(
                    f"Validation Error: Video file not found for JSON file.\n"
                    f"Missing video: {video_path}\n"
                    f"Corresponding JSON: {json_path}"
                )

            with json_path.open("rb") as f:
                json_data = orjson.loads(f.read())

            bbox_data = json_data["bboxes"]
            wlw_data = json_data["wlw"]

            yield {
                "video": str(video_path),
                "bboxes": bbox_data,
                "wlw": wlw_data,
            }


features = Features(
    {
        "video": Video(decode=False),
        "bboxes": Sequence(Sequence(Sequence(Value("float32"), length=4))),
        "wlw": Sequence(Sequence(Sequence(Value("bool")))),
    }
)

data_directory = "/local_scratch/gzappavi/datasets/wlw_dataset"

data_dir_path = Path(data_directory)
subdirectories = sorted(p for p in data_dir_path.iterdir() if p.is_dir())
gen_kwargs = {"folder_paths": subdirectories}

my_dataset = Dataset.from_generator(
    dataset_generator,
    gen_kwargs=gen_kwargs,
    features=features,
    num_proc=cpu_count(),
)

In [3]:
from itertools import permutations

def find_true_stretches(bool_array):
    if bool_array.ndim != 3:
        raise ValueError("Input array must be 3-dimensional")

    k1, k2, _ = bool_array.shape
    assert k1 == k2
    result = {}


    for i, j in permutations(range(k1), r=2):
        # Extract the 1D slice
        slice_1d = bool_array[i, j]

        # Pad the slice with False at both ends and find differences
        padded_slice = np.concatenate(([False], slice_1d, [False]))
        diffs = np.diff(padded_slice.astype(int))

        # Find the start and end indices of True stretches
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0]

        if starts.size > 0:
            result[(i, j)] = list(zip(starts, ends))
        else:
            result[(i, j)] = []

    return result


In [ ]:
from collections import Counter

counter = Counter()
max_len = 0

for i, sample in enumerate(my_dataset):
    wlw_matrix = np.array(sample["wlw"])
    wlw_matrix = wlw_matrix.transpose(1, 2, 0)

    ranges_dict = find_true_stretches(wlw_matrix)

    for ranges in ranges_dict.values():
        max_len = max(len(ranges), max_len)
        last = 0
        for a, b in ranges:
            assert b > a
            assert last == 0 or a > 0
            counter[a - last] += 1
            counter[b - a] += 1
            last = b

print(max_len)
print(len(counter))

86
617


In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F


SPECIAL_TOKENS = [
    "[PAD]",
    "[INDEX_I]",
    "[INDEX_J]",
    "[START_POS]",
    "[WIDTH]",
    "[GAP]",
    "[SEP]",
    "[EOS]",
]


In [52]:
def tokenize_ranges(ranges: list[tuple[int, int]]) -> list[int]:
    """
    Helper function to tokenize only the range list for a single pair.
    (This is a simplified version of our previous main tokenizer)
    """
    if not ranges:
        return []

    parts = []
    # Process first range (start_pos, width)
    a1, b1 = ranges[0]
    parts.append("[START_POS]")
    parts.extend(str(a1))
    parts.append("[WIDTH]")
    parts.extend(str(b1 - a1))

    # Process subsequent ranges (gap, width)
    for k in range(len(ranges) - 1):
        _, b_curr = ranges[k]
        a_next, b_next = ranges[k+1]
        parts.append("[GAP]")
        parts.extend(str(a_next - b_curr))
        parts.append("[WIDTH]")
        parts.extend(str(b_next - a_next))

    return parts

In [71]:
seq_list = []
empty_count = 0

for sample in my_dataset:
    wlw_matrix = np.array(sample["wlw"])
    wlw_matrix = wlw_matrix.transpose(1, 2, 0)

    ranges_dict = find_true_stretches(wlw_matrix)
    seq_parts = []

    ranges_found = False

    for (i, j), ranges in ranges_dict.items():
        seq_parts.append("[INDEX_I]")
        seq_parts.extend(str(i))
        seq_parts.append("[INDEX_J]")
        seq_parts.extend(str(j))

        if ranges:
            ranges_found = True

        ranges_parts = tokenize_ranges(ranges)
        if ranges and not ranges_parts:
            raise Exception
        seq_parts.extend(ranges_parts)
        seq_parts.append("[SEP]")

    if not ranges_found:
        empty_count += 1

    part = seq_parts.pop()
    assert part == "[SEP]"
    seq_parts.append("[EOS]")
    seq_list.append(" ".join(seq_parts))


In [67]:
empty_count / (len(seq_list) + empty_count)

0.29929347724773037

In [68]:
seq_list[123]

'[INDEX_I] 0 [INDEX_J] 1 [START_POS] 3 6 [WIDTH] 5 [GAP] 1 [WIDTH] 2 [SEP] [INDEX_I] 1 [INDEX_J] 0 [START_POS] 0 [WIDTH] 6 5 [GAP] 1 [WIDTH] 1 1 [GAP] 1 [WIDTH] 5 [GAP] 9 [WIDTH] 1 [EOS]'

In [72]:
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer


tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
tokenizer.add_special_tokens(list(SPECIAL_TOKENS))

trainer = WordLevelTrainer(special_tokens=list(SPECIAL_TOKENS))
# trainer = WordLevelTrainer()
tokenizer.train_from_iterator(seq_list, trainer=trainer)

In [73]:
sorted(tokenizer.get_vocab().items(), key=lambda x: x[1])

[('[PAD]', 0),
 ('[INDEX_I]', 1),
 ('[INDEX_J]', 2),
 ('[START_POS]', 3),
 ('[WIDTH]', 4),
 ('[GAP]', 5),
 ('[SEP]', 6),
 ('[EOS]', 7),
 ('1', 8),
 ('0', 10),
 ('2', 14),
 ('3', 16),
 ('4', 17),
 ('5', 20),
 ('6', 21),
 ('7', 22),
 ('8', 23),
 ('9', 24)]